# Silver Layer — microbiologyevents

## READINESS: profiled, rules drafted, awaiting remaining profiling and team approval

The profiling cells below were run (results recorded in **Profiling Findings**) and the
Silver rules derived from them are implemented. A few checks added in this version are
marked *not yet profiled* and must be run and recorded before approval. Until the
`profiling_approved` widget is set to `true`, the publish step raises an error and nothing
is written. The dependency-ordered runner excludes this notebook by default.

## Source
`<source_schema>.microbiologyevents` (default `cdr_mimiciv.bronze.microbiologyevents`)

## Target
`<target_schema>.microbiologyevents` (default `cdr_mimiciv.silver.microbiologyevents`),
published with CREATE OR REPLACE semantics. No tables are dropped.

## Grain (official MIMIC-IV hosp documentation, `microbiologyevents`)
One row per microbiology result line, identified by `microevent_id` ("a unique integer
denoting the row"). The documentation describes a hierarchy in a single table: a specimen
(`micro_specimen_id`, with `spec_itemid` / `spec_type_desc`) has one or more tests
(`test_seq`, `test_itemid` / `test_name`); if an organism grows, each organism is a row
(`org_itemid` / `org_name`, `isolate_num`); if antibiotics are tested on an organism, each
antibiotic is a further row (`ab_itemid` / `ab_name`, `dilution_text`,
`dilution_comparison`, `dilution_value`, `interpretation`).
Other documented fields: `subject_id`, `hadm_id` (assigned from the transfers table, so
some specimens near a stay have no `hadm_id`; a time-based join to admissions is the
documented alternative), `order_provider_id`, `chartdate` / `charttime` (`charttime` is
NULL when no time is available; `chartdate` is always present and equals the truncated
`charttime` when both exist), `storedate` / `storetime` (time of the last known update of
the result), `quantity`, `comments` (`___` is a fully de-identified comment; NULL means no
comment). `interpretation` is documented as "S" sensitive, "R" resistant, "I"
intermediate, "P" pending.
[DOC] If no organism grew, `org_name` is NULL (a negative culture), and organism and
antibiotic fields are NULL by structure. However, itemid 90856 carries the value
"NEGATIVE" and must be considered when separating positive from negative findings.
Documented NOT NULL: `microevent_id`, `subject_id`, `micro_specimen_id`, `chartdate`,
`spec_itemid`, `spec_type_desc`, `test_seq`.

## Silver transformation (derived from the profiling findings)
Principle, as in admissions and labevents: preserve source value, identify anomaly, add a
derived column or DQ flag, let downstream users decide.
- Select every Bronze column unchanged, in its Bronze type and order.
- Add derived column (our engineering choice): `dilution_comparison_std` (trimmed; source
  values are right-padded with spaces, e.g. `"<=        "`; empty string to NULL; `=>` is
  kept as written, not rewritten to `>=`).
- Add DQ flags (our engineering choice), all non-null booleans:
  - `dq_result_not_reported`: `org_name = 'CANCELLED'` (41,504 rows). The organism column
    carries a cancelled-test marker, not an organism. Same name and meaning as labevents.
  - `dq_interpretation_undocumented`: `interpretation` present but not one of the
    documented S / R / I / P (285 rows: D 217, Z 56, N 12).
  - `dq_storedate_before_chartdate`: `storedate < chartdate` (35 rows).
  - `dq_missing_spec_type_desc`: documented NOT NULL column is NULL (1 row).
  - `dq_hadm_not_in_admissions`: non-null `(subject_id, hadm_id)` not in Silver admissions
    (1 distinct pair). Same name and meaning as labevents.
- Add `_silver_processed_at`.
- No deduplication, no NULL filling, no timestamp correction, no row removal. NULL
  `hadm_id` is documented behaviour and is not imputed or flagged; NULL organism and
  antibiotic fields are structural and are not filled. No positive/negative culture
  indicator, resistance summary or specimen roll-up is computed: those are Gold definitions.

## Open profiling questions
1. Do `dilution_text`, `dilution_comparison` and `dilution_value` agree?
2. Where does itemid 90856 ("NEGATIVE") appear, and in which column?
3. Are there other `org_name` markers of a cancelled or unreported test besides
   `CANCELLED` (only the top 100 names were inspected)?
4. How many rows carry the one `(subject_id, hadm_id)` pair not found in admissions?

## Cost note
About 4 million rows. Profiling uses one aggregate pass plus a few grouped scans; relationship
checks use distinct keys; no `exceptAll` is used; the published table is checked with one
aggregate pass.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.microbiologyevents"
TARGET_TABLE = f"{SILVER}.microbiologyevents"
KEY = "microevent_id"

DOCUMENTED_COLUMNS = [
    "microevent_id", "subject_id", "hadm_id", "micro_specimen_id", "order_provider_id",
    "chartdate", "charttime", "spec_itemid", "spec_type_desc", "test_seq",
    "storedate", "storetime", "test_itemid", "test_name", "org_itemid", "org_name",
    "isolate_num", "quantity", "ab_itemid", "ab_name", "dilution_text",
    "dilution_comparison", "dilution_value", "interpretation", "comments"
]

# [DOC] documented interpretation codes
DOCUMENTED_INTERPRETATION = ["S", "R", "I", "P"]

# org_name values that mark a cancelled / unreported test rather than an organism
# (observed in profiling; explicit list, auditable)
NOT_REPORTED_ORG_NAMES = ["CANCELLED"]

In [0]:
micro_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = micro_bronze.columns

micro_bronze.printSchema()
print("Documented columns missing from Bronze:", [c for c in DOCUMENTED_COLUMNS if c not in SOURCE_COLUMNS])
print("Bronze columns not in documentation:", [c for c in SOURCE_COLUMNS if c not in DOCUMENTED_COLUMNS])
display(micro_bronze.limit(20))

root
 |-- microevent_id: integer (nullable = true)
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- micro_specimen_id: integer (nullable = true)
 |-- order_provider_id: string (nullable = true)
 |-- chartdate: timestamp (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- spec_itemid: integer (nullable = true)
 |-- spec_type_desc: string (nullable = true)
 |-- test_seq: integer (nullable = true)
 |-- storedate: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- test_itemid: integer (nullable = true)
 |-- test_name: string (nullable = true)
 |-- org_itemid: integer (nullable = true)
 |-- org_name: string (nullable = true)
 |-- isolate_num: integer (nullable = true)
 |-- quantity: string (nullable = true)
 |-- ab_itemid: integer (nullable = true)
 |-- ab_name: string (nullable = true)
 |-- dilution_text: string (nullable = true)
 |-- dilution_comparison: string (nullable = true)
 |-- dilution_value: double (nu

microevent_id,subject_id,hadm_id,micro_specimen_id,order_provider_id,chartdate,charttime,spec_itemid,spec_type_desc,test_seq,storedate,storetime,test_itemid,test_name,org_itemid,org_name,isolate_num,quantity,ab_itemid,ab_name,dilution_text,dilution_comparison,dilution_value,interpretation,comments
1,10000032,null,1304715,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70046,IMMUNOLOGY,1,2180-03-26T00:00:00.000Z,2180-03-26T10:54:00.000Z,90123,HCV VIRAL LOAD,null,null,null,null,null,null,null,null,null,null,___
2,10000032,null,3342526,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70093,Blood (Toxo),1,2180-03-26T00:00:00.000Z,2180-03-26T10:17:00.000Z,90144,TOXOPLASMA IgG ANTIBODY,null,null,null,null,null,null,null,null,null,null,"NEGATIVE FOR TOXOPLASMA IgG ANTIBODY BY EIA. 0.0 IU/ML. Reference Range: Negative < 4 IU/ml, Positive >= 8 IU/ml. If acute infection is suspected request IgM antibody testing and/or submit convalescent serum in 2-3 weeks."
3,10000032,null,3910370,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70017,SEROLOGY/BLOOD,1,2180-03-24T00:00:00.000Z,2180-03-24T05:27:00.000Z,90248,RAPID PLASMA REAGIN TEST,null,null,null,null,null,null,null,null,null,null,NONREACTIVE. Reference Range: Non-Reactive.
4,10000032,null,5401234,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70017,SEROLOGY/BLOOD,1,2180-03-26T00:00:00.000Z,2180-03-26T13:38:00.000Z,90091,VARICELLA-ZOSTER IgG SEROLOGY,null,null,null,null,null,null,null,null,null,null,POSITIVE BY EIA. A positive IgG result generally indicates past exposure and/or immunity.
5,10000032,null,6287540,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70046,IMMUNOLOGY,1,2180-03-26T00:00:00.000Z,2180-03-26T10:25:00.000Z,90175,HIV-1 Viral Load/Ultrasensitive,null,null,null,null,null,null,null,null,null,null,"HIV-1 RNA is not detected. Performed using the ___ HIV-1 Test v2.0. Detection Range: 20-10,000,000 copies/mL. This test is approved for monitoring HIV-1 viral load in known HIV-positive patients. It is not approved for diagnosis of acute HIV infection. In symptomatic acute HIV infection (acute retroviral syndrome), the viral load is usually very high (>>1000 copies/mL). If acute HIV infection is clinically suspected and there is a detectable but low viral load, please contact the laboratory for interpretation. It is recommended that any NEW positive HIV-1 viral load result, in the absence of positive serology, be confirmed by submitting a new sample FOR HIV-1 PCR, in addition to serological testing."
6,10000032,null,7322675,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70017,SEROLOGY/BLOOD,1,2180-03-24T00:00:00.000Z,2180-03-24T11:50:00.000Z,90254,RUBELLA IgG SEROLOGY,null,null,null,null,null,null,null,null,null,null,POSITIVE BY EIA. A positive IgG result generally indicates past exposure and/or immunity.
7,10000032,null,8934175,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70017,SEROLOGY/BLOOD,1,2180-03-24T00:00:00.000Z,2180-03-24T12:40:00.000Z,90127,"RUBEOLA ANTIBODY, IgG",null,null,null,null,null,null,null,null,null,null,POSITIVE BY EIA. A positive IgG result generally indicates past exposure and/or immunity.
8,10000032,null,9041996,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70088,Blood (EBV),1,2180-03-25T00:00:00.000Z,2180-03-25T11:54:00.000Z,90101,EPSTEIN-BARR VIRUS VCA-IgG AB,null,null,null,null,null,null,null,null,null,null,POSITIVE BY EIA.
9,10000032,null,9041996,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70088,Blood (EBV),2,2180-03-25T00:00:00.000Z,2180-03-25T11:54:00.000Z,90135,EPSTEIN-BARR VIRUS EBNA IgG AB,null,null,null,null,null,null,null,null,null,null,POSITIVE BY EIA.
10,10000032,null,9041996,P69FQC,2180-03-23T00:00:00.000Z,2180-03-23T11:51:00.000Z,70088,Blood (EBV),3,2180-03-25T00:00:00.000Z,2180-03-25T12:46:00.000Z,90134,EPSTEIN-BARR VIRUS VCA-IgM AB,null,null,null,null,null,null,null,null,null,null,"NEGATIVE <1:10 BY IFA. INTERPRETATION: RESULTS INDICATIVE OF PAST EBV

In [0]:
# single-pass profile: grain/key, missingness of every column, timestamp order
def _cnt(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

profile_exprs = [
    F.count("*").alias("total_rows"),
    F.countDistinct(KEY).alias("distinct_microevent_id"),
]
profile_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
profile_exprs += [
    _cnt(F.col("storetime") < F.col("charttime")).alias("storetime_before_charttime"),
    _cnt(F.col("storedate") < F.col("chartdate")).alias("storedate_before_chartdate"),
    _cnt(F.col("charttime").isNotNull() & (F.to_date("charttime") != F.to_date("chartdate")))
        .alias("charttime_date_differs_from_chartdate"),
    _cnt(F.col("interpretation").isNotNull() & ~F.col("interpretation").isin(DOCUMENTED_INTERPRETATION))
        .alias("interpretation_undocumented"),
    _cnt(F.col("org_name").isin(NOT_REPORTED_ORG_NAMES)).alias("org_name_not_reported"),
]

micro_profile = micro_bronze.agg(*profile_exprs).first().asDict()
for k, v in micro_profile.items():
    print(f"{k}: {v}")

total_rows: 3988224
distinct_microevent_id: 3988224
null_microevent_id: 0
null_subject_id: 0
null_hadm_id: 2228343
null_micro_specimen_id: 0
null_order_provider_id: 2839884
null_chartdate: 0
null_charttime: 303456
null_spec_itemid: 0
null_spec_type_desc: 1
null_test_seq: 0
null_storedate: 18540
null_storetime: 32192
null_test_itemid: 0
null_test_name: 0
null_org_itemid: 2352859
null_org_name: 2352859
null_isolate_num: 2352859
null_quantity: 3988042
null_ab_itemid: 2577966
null_ab_name: 2577966
null_dilution_text: 2613366
null_dilution_comparison: 2614476
null_dilution_value: 2614476
null_interpretation: 2577966
null_comments: 1269864
storetime_before_charttime: 0
storedate_before_chartdate: 35
charttime_date_differs_from_chartdate: 0
interpretation_undocumented: 285
org_name_not_reported: 41504


In [0]:
# duplicates: key duplicates, exact duplicate rows, and the documented hierarchy key
non_null_ids = micro_profile["total_rows"] - micro_profile[f"null_{KEY}"]
print("microevent_id duplicates present:", micro_profile["distinct_microevent_id"] != non_null_ids)

exact_duplicate_groups = micro_bronze.groupBy(*SOURCE_COLUMNS).count().filter(F.col("count") > 1)
print("Exact duplicate row groups:", exact_duplicate_groups.count())

HIERARCHY_KEY = ["micro_specimen_id", "test_seq", "isolate_num", "ab_itemid"]
hierarchy_dups = micro_bronze.groupBy(*HIERARCHY_KEY).count().filter(F.col("count") > 1)
print(f"Duplicate groups on {HIERARCHY_KEY} (informational; NULL members group together):", hierarchy_dups.count())

microevent_id duplicates present: False
Exact duplicate row groups: 0
Duplicate groups on ['micro_specimen_id', 'test_seq', 'isolate_num', 'ab_itemid'] (informational; NULL members group together): 81295


In [0]:
# categorical values
for c in ["interpretation", "dilution_comparison", "quantity", "spec_type_desc", "test_name", "org_name", "ab_name"]:
    print(f"=== {c} (top 100) ===")
    display(micro_bronze.groupBy(c).count().orderBy(F.desc("count")).limit(100))

=== interpretation (top 100) ===


interpretation,count
null,2577966
S,1125203
R,244726
I,40040
D,217
Z,56
N,12
P,4


=== dilution_comparison (top 100) ===


dilution_comparison,count
null,2614476
<=,948535
=,221948
=>,203265


=== quantity (top 100) ===


quantity,count
null,3988042
">100,000 CFU/mL",51
"10,000-100,000 CFU/mL",44
SPARSE GROWTH,34
MODERATE,20
RARE,12
1 COLONY,7
MODERATE-NUMBER,5
FEW,4
1,4


... (4 outputs left out)

=== org_name (top 100) ===


org_name,count
null,2352859
ESCHERICHIA COLI,579398
STAPH AUREUS COAG +,219582
KLEBSIELLA PNEUMONIAE,155688
PSEUDOMONAS AERUGINOSA,83290
PROTEUS MIRABILIS,68163
"STAPHYLOCOCCUS, COAGULASE NEGATIVE",58412
ENTEROCOCCUS SP.,53387
CANCELLED,41504
YEAST,33959


=== ab_name (top 100) ===


ab_name,count
null,2577966
GENTAMICIN,128193
TRIMETHOPRIM/SULFA,109318
CIPROFLOXACIN,90054
CEFTAZIDIME,90011
TOBRAMYCIN,89458
MEROPENEM,89383
CEFEPIME,89237
CEFTRIAXONE,79640
PIPERACILLIN/TAZO,76849


In [0]:
# structural missingness: organism / antibiotic NULLs against each other
display(
    micro_bronze
    .groupBy(
        F.col("org_itemid").isNull().alias("org_null"),
        F.col("ab_itemid").isNull().alias("ab_null"),
        F.col("interpretation").isNull().alias("interpretation_null"),
        F.col("dilution_value").isNull().alias("dilution_value_null")
    )
    .count()
    .orderBy(F.desc("count"))
)

org_null,ab_null,interpretation_null,dilution_value_null,count
true,true,true,true,2352859
false,false,false,false,1373748
false,true,true,true,225107
false,false,false,true,36510


In [0]:
# dilution agreement: does dilution_text equal comparison + value?
dilution_text_num = F.regexp_extract(F.trim("dilution_text"), r"(-?[0-9]+(\.[0-9]+)?)", 1)
display(
    micro_bronze.where(F.col("dilution_text").isNotNull() | F.col("dilution_value").isNotNull())
    .agg(
        _cnt(F.col("dilution_text").isNotNull() & F.col("dilution_value").isNull()).alias("text_without_value"),
        _cnt(F.col("dilution_text").isNull() & F.col("dilution_value").isNotNull()).alias("value_without_text"),
        _cnt(
            F.col("dilution_value").isNotNull() & (dilution_text_num != "")
            & (dilution_text_num.cast("double") != F.col("dilution_value"))
        ).alias("text_number_differs_from_value"),
        _cnt(
            F.col("dilution_comparison").isNotNull()
            & ~F.trim("dilution_text").startswith(F.trim("dilution_comparison"))
        ).alias("text_does_not_start_with_comparison"),
    )
)
display(
    micro_bronze.where(F.col("dilution_text").isNotNull() & F.col("dilution_value").isNull())
    .groupBy("dilution_text").count().orderBy(F.desc("count")).limit(30)
)

text_without_value,value_without_text,text_number_differs_from_value,text_does_not_start_with_comparison
1110,0,0,221948


dilution_text,count
>256,692
>16,158
>8,77
>4,70
>32,46
>2,36
>128,16
>64,13
>1,2


In [0]:
# documented "NEGATIVE" item 90856: where does it appear?
display(
    micro_bronze
    .where(
        (F.col("spec_itemid") == 90856) | (F.col("test_itemid") == 90856)
        | (F.col("org_itemid") == 90856) | (F.col("ab_itemid") == 90856)
    )
    .groupBy("spec_itemid", "test_itemid", "test_name", "org_itemid", "org_name")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

spec_itemid,test_itemid,test_name,org_itemid,org_name,count
70064,90233,C. difficile Toxin antigen assay,90856,NEGATIVE,327
70064,90221,C. difficile PCR,90856,NEGATIVE,26


In [0]:
# itemid / name consistency: more than one name per itemid
for id_col, name_col in [("spec_itemid", "spec_type_desc"), ("test_itemid", "test_name"),
                         ("org_itemid", "org_name"), ("ab_itemid", "ab_name")]:
    inconsistent = (
        micro_bronze.where(F.col(id_col).isNotNull())
        .groupBy(id_col)
        .agg(F.countDistinct(name_col).alias("names"))
        .filter(F.col("names") > 1)
    )
    print(f"{id_col} with more than one {name_col}:", inconsistent.count())

spec_itemid with more than one spec_type_desc: 0
test_itemid with more than one test_name: 0
org_itemid with more than one org_name: 0
ab_itemid with more than one ab_name: 0


In [0]:
# relationships (observational only; findings are recorded, not turned into flags here)
patients_table = f"{SILVER}.patients"
admissions_table = f"{SILVER}.admissions"

for _t in (patients_table, admissions_table):
    assert spark.catalog.tableExists(_t), f"Missing reference: {_t}"

patients_ref = spark.table(patients_table).select("subject_id").distinct()
print(
    "distinct subject_id not found in patients:",
    micro_bronze.select("subject_id").distinct().join(patients_ref, "subject_id", "left_anti").count()
)

admissions_ref = spark.table(admissions_table).select("subject_id", "hadm_id").distinct()
orphan_pairs = (
    micro_bronze.where(F.col("hadm_id").isNotNull())
    .groupBy("subject_id", "hadm_id").count()
    .join(admissions_ref, ["subject_id", "hadm_id"], "left_anti")
)
print("distinct (subject_id, hadm_id) with non-null hadm_id not found in admissions:", orphan_pairs.count())
display(orphan_pairs)

distinct subject_id not found in patients: 0
distinct (subject_id, hadm_id) with non-null hadm_id not found in admissions: 1


subject_id,hadm_id,count
17080681,25029676,3


## Profiling Findings

Recorded from the profiling run of 2026-10-19 on `cdr_mimiciv.bronze.microbiologyevents`
(saved outputs of the previous notebook version). Rows marked *not yet profiled* must be
filled from the cells above before `profiling_approved` is set.
[DOC] = stated in official MIMIC documentation; [OURS] = our engineering choice.

| Check | Result | Silver treatment |
|---|---:|---|
| Rows / distinct `microevent_id` | 3,988,224 / 3,988,224 | Key unique; no deduplication |
| Exact duplicate rows | 0 groups | None |
| Duplicates on (`micro_specimen_id`, `test_seq`, `isolate_num`, `ab_itemid`) | 81,295 groups | Informational; not the documented key (NULL members group together) |
| NULL `microevent_id`, `subject_id`, `micro_specimen_id`, `chartdate`, `spec_itemid`, `test_seq`, `test_itemid`, `test_name` | 0 | None |
| NULL `spec_type_desc` | 1 | `dq_missing_spec_type_desc` [OURS]; [DOC] NOT NULL |
| NULL `hadm_id` | 2,228,343 (55.9%) | Keep NULL, no flag; [DOC] assigned via transfers |
| NULL `charttime` | 303,456 (7.6%) | Keep; [DOC] no time available, `chartdate` holds the date |
| NULL `order_provider_id` / `storedate` / `storetime` / `comments` | 2,839,884 / 18,540 / 32,192 / 1,269,864 | Keep |
| NULL `quantity` | 3,988,042 (182 values present) | Keep; free-text growth descriptions, not parsed |
| Organism / antibiotic NULL pattern | none 2,352,859; all present 1,373,748; organism only 225,107; no `dilution_value` 36,510 | Structural; [DOC] no growth or no sensitivity test; not filled |
| NULL `dilution_text` / `dilution_comparison` = `dilution_value` | 2,613,366 / 2,614,476 (1,110 text without value) | Keep; agreement *not yet profiled* |
| `interpretation` values | S 1,125,203; R 244,726; I 40,040; D 217; Z 56; N 12; P 4 | Values kept; D/Z/N (285) flagged `dq_interpretation_undocumented` [OURS; DOC lists S/R/I/P] |
| `dilution_comparison` values | `<=` 948,535; `=` 221,948; `=>` 203,265, all right-padded with spaces | `dilution_comparison_std` trimmed [OURS] |
| `org_name = 'CANCELLED'` | 41,504 | `dq_result_not_reported` [OURS]; row kept |
| itemid 90856 "NEGATIVE" | *not yet profiled* | No change; [DOC] consider in positive/negative logic (Gold) |
| `ab_name` spellings | e.g. `VORICONZAOLE` 1, `DELAFLOXICIN` 3 | Source spelling kept |
| itemid / name consistency (spec, test, org, ab) | 0 itemids with more than one name | None |
| `storetime` before `charttime` | 0 | None |
| `storedate` before `chartdate` | 35 | `dq_storedate_before_chartdate` [OURS]; no correction |
| `charttime` date differs from `chartdate` | 0 | None; [DOC] chartdate is truncated charttime |
| `subject_id` not in patients | 0 distinct | None |
| `(subject_id, hadm_id)` not in admissions | 1 distinct pair (row count *not yet profiled*) | `dq_hadm_not_in_admissions` [OURS]; row kept |

In [0]:
# Silver transformation: all source columns unchanged + derived column + DQ flags
# (our engineering choices, see Profiling Findings); no rows removed, no source value changed

# distinct admission keys from Silver admissions (small; drives dq_hadm_not_in_admissions)
admissions_keys = (
    spark.table(f"{SILVER}.admissions")
    .select("subject_id", "hadm_id").distinct()
    .withColumn("_adm_match", F.lit(True))
)

comparison_trim = F.trim(F.col("dilution_comparison"))

micro_silver = (
    micro_bronze.select(*SOURCE_COLUMNS)
    .join(F.broadcast(admissions_keys), ["subject_id", "hadm_id"], "left")
    .select(
        *SOURCE_COLUMNS,
        F.when(comparison_trim == "", F.lit(None).cast("string"))
            .otherwise(comparison_trim)
            .alias("dilution_comparison_std"),
        F.coalesce(F.col("org_name").isin(NOT_REPORTED_ORG_NAMES), F.lit(False))
            .alias("dq_result_not_reported"),
        F.coalesce(
            F.col("interpretation").isNotNull() & ~F.col("interpretation").isin(DOCUMENTED_INTERPRETATION),
            F.lit(False)
        ).alias("dq_interpretation_undocumented"),
        F.coalesce(F.col("storedate") < F.col("chartdate"), F.lit(False))
            .alias("dq_storedate_before_chartdate"),
        F.col("spec_type_desc").isNull().alias("dq_missing_spec_type_desc"),
        (F.col("hadm_id").isNotNull() & F.col("_adm_match").isNull())
            .alias("dq_hadm_not_in_admissions"),
        F.current_timestamp().alias("_silver_processed_at"),
    )
)

DQ_FLAGS = [
    "dq_result_not_reported", "dq_interpretation_undocumented",
    "dq_storedate_before_chartdate", "dq_missing_spec_type_desc", "dq_hadm_not_in_admissions",
]

expected_schema = {f.name: f.dataType.simpleString() for f in micro_bronze.schema.fields}
expected_schema.update({"dilution_comparison_std": "string"})
expected_schema.update({c: "boolean" for c in DQ_FLAGS})
expected_schema.update({"_silver_processed_at": "timestamp"})

# profiled expected counts (2026-10-19 run); add dq_hadm_not_in_admissions once its row count is profiled
PROFILED_COUNTS = {
    "silver_rows": 3988224,
    "dq_result_not_reported": 41504,
    "dq_interpretation_undocumented": 285,
    "dq_storedate_before_chartdate": 35,
    "dq_missing_spec_type_desc": 1,
}

micro_silver.printSchema()

root
 |-- microevent_id: integer (nullable = true)
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- micro_specimen_id: integer (nullable = true)
 |-- order_provider_id: string (nullable = true)
 |-- chartdate: timestamp (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- spec_itemid: integer (nullable = true)
 |-- spec_type_desc: string (nullable = true)
 |-- test_seq: integer (nullable = true)
 |-- storedate: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- test_itemid: integer (nullable = true)
 |-- test_name: string (nullable = true)
 |-- org_itemid: integer (nullable = true)
 |-- org_name: string (nullable = true)
 |-- isolate_num: integer (nullable = true)
 |-- quantity: string (nullable = true)
 |-- ab_itemid: integer (nullable = true)
 |-- ab_name: string (nullable = true)
 |-- dilution_text: string (nullable = true)
 |-- dilution_comparison: string (nullable = true)
 |-- dilution_value: double (nu

In [0]:
# publish gate: nothing below runs until profiling has been reviewed and approved
if not PROFILING_APPROVED:
    raise RuntimeError(
        "microbiologyevents is not approved for publication: run and review the profiling "
        "cells, fill the 'not yet profiled' findings and any justified rules, then set widget "
        "profiling_approved=true."
    )

# cheap pre-publish validation (no data scan)
pre_schema = {f.name: f.dataType.simpleString() for f in micro_silver.schema.fields}
assert list(pre_schema.items()) == list(expected_schema.items()), "Unexpected Silver schema"
assert KEY in pre_schema, f"Key column {KEY} missing"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    micro_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.microbiologyevents


In [0]:
# post-publish validation: one aggregate pass over Silver, one count of Bronze (no exceptAll)
published = spark.table(TARGET_TABLE)

post = published.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct(KEY).alias("distinct_key"),
    _cnt(F.col(KEY).isNull()).alias("null_key"),
    _cnt(F.col("_silver_processed_at").isNull()).alias("null_processed_at"),
    _cnt(F.greatest(*[F.col(c).isNull() for c in DQ_FLAGS])).alias("null_dq_flags"),
    *[_cnt(F.col(c)).alias(c) for c in DQ_FLAGS],
    _cnt(
        F.col("dilution_comparison_std").isNotNull()
        & (F.col("dilution_comparison_std") != F.trim("dilution_comparison"))
    ).alias("comparison_std_mismatches"),
).first().asDict()
bronze_rows = micro_bronze.count()

published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

print("Bronze rows:", bronze_rows)
print("Published:", post)

assert post["silver_rows"] == bronze_rows, "Silver row count differs from Bronze"
assert post["null_key"] == 0, f"{KEY} contains NULLs"
# microevent_id is the documented primary key; profiling confirmed it is unique
assert post["distinct_key"] == post["silver_rows"], f"{KEY} is not unique"
assert post["null_processed_at"] == 0
assert post["null_dq_flags"] == 0, "DQ flags must be non-null booleans"
assert post["comparison_std_mismatches"] == 0, "dilution_comparison_std differs from trimmed source"
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"

if ENFORCE_PROFILED_COUNTS:
    for name, expected in PROFILED_COUNTS.items():
        assert post[name] == expected, f"{name}: expected {expected}, got {post[name]}"

dbutils.notebook.exit("microbiologyevents:OK")